# Lab 04: Public-Key Cryptography — Diffie-Hellman & RSA Mechanics
**Student Name:** Md. Akib Al Jawad  
**Student ID:** IT-23020  
**Department:** Information and Communication Technology (ICT)  
**Institution:** Mawlana Bhashani Science and Technology University (MBSTU)  
**Reference:** IIT Bombay Maths and Physics Club Workshop — Demo Session Notebook

---

## Overview
Asymmetric cryptography solves the fundamental problem of key distribution over insecure channels. This notebook explores:
1. **Diffie-Hellman Key Exchange Protocol**: Solving key agreement using the Discrete Logarithm Problem (DLP) in cyclic groups $(\mathbb{Z}_p^*, \times)$.
2. **The RSA Cryptosystem**: Key generation, Euler's totient function $\phi(N) = (p - 1)(q - 1)$, encryption, decryption, and algebraic proof of correctness via Euler's theorem.
3. **Text Encoding**: Base-256 serialization of arbitrary text into big integers.
4. **Quantum Threat (Shor's Algorithm)**: Why classical discrete log and integer factorization will be solved in polynomial time by quantum computers.



### 1. Diffie-Hellman Key Agreement Protocol
Alice and Bob agree publicly on a prime modulus $p$ and a generator $g$ of $\mathbb{Z}_p^*$.
1. Alice chooses secret private key $a \in [1, p-1]$ and sends $A = g^a \bmod p$.
2. Bob chooses secret private key $b \in [1, p-1]$ and sends $B = g^b \bmod p$.
3. Alice computes $K_A = B^a \bmod p = (g^b)^a \equiv g^{ab} \pmod p$.
4. Bob computes $K_B = A^b \bmod p = (g^a)^b \equiv g^{ab} \pmod p$.
5. Both share the synthesized key $K = g^{ab} \pmod p$, while Eve only sees $p, g, A, B$.



In [1]:
p = 23    # Modulus (prime)
g = 5     # Generator

# Private keys
a = 6     # Alice
b = 15    # Bob

# Public transmissions
A = pow(g, a, p)
B = pow(g, b, p)
print(f"Public Parameters: p = {p}, g = {g}")
print(f"Alice's Public Key A = {A}")
print(f"Bob's Public Key   B = {B}")

# Shared secret computation
shared_Alice = pow(B, a, p)
shared_Bob   = pow(A, b, p)
assert shared_Alice == shared_Bob
print(f"Synthesized Shared Secret: {shared_Alice} (Both Alice and Bob agree!)")



Public Parameters: p = 23, g = 5


Alice's Public Key A = 8


Bob's Public Key   B = 19


Synthesized Shared Secret: 2 (Both Alice and Bob agree!)


### 2. RSA Cryptosystem Mechanics
#### Mathematical Proof of Correctness:
Let $N = pq$ where $p, q$ are distinct primes. Then $\phi(N) = (p - 1)(q - 1)$.  
Public key $(e, N)$ and private key $(d, N)$ satisfy:
$$ed \equiv 1 \pmod{\phi(N)} \implies ed = 1 + k\phi(N) = 1 + k(p - 1)(q - 1)$$
Encryption: $c = m^e \bmod N$.  
Decryption: $m' = c^d \bmod N = m^{ed} \bmod N$.

If $\gcd(m, p) = 1$, by Fermat's Little Theorem:
$$m^{ed} = m^{1 + k(p-1)(q-1)} = m \cdot (m^{p-1})^{k(q-1)} \equiv m \cdot 1 \equiv m \pmod p$$
If $p \mid m$, then $m \equiv 0 \pmod p$, so $m^{ed} \equiv 0 \equiv m \pmod p$.  
Thus $m^{ed} \equiv m \pmod p$ always holds. By identical symmetry, $m^{ed} \equiv m \pmod q$.  
Since $\gcd(p, q) = 1$, by the Chinese Remainder Theorem:
$$m^{ed} \equiv m \pmod{pq} \implies m' \equiv m \pmod N$$



In [1]:
def extended_euclid(a, b):
    if b == 0: return (1, 0)
    x1, y1 = extended_euclid(b, a % b)
    return (y1, x1 - (a // b) * y1)

def invert_modulo(a, n):
    b, _ = extended_euclid(a, n)
    return (b % n + n) % n

def convert_to_int(text):
    res = 0
    for ch in text:
        res = res * 256 + ord(ch)
    return res

def convert_to_str(n):
    res = []
    while n > 0:
        res.append(chr(n % 256))
        n //= 256
    return "".join(reversed(res))

# RSA Key Generation
p = 1000000007
q = 1000000009
N = p * q
phi = (p - 1) * (q - 1)
e = 65537
d = invert_modulo(e, phi)

message = "HELLO RSA"
m_int = convert_to_int(message)
c_int = pow(m_int, e, N)
decrypted_int = pow(c_int, d, N)
decrypted_msg = convert_to_str(decrypted_int)

print(f"Modulus N:        {N}")
print(f"Public Exponent:  {e}")
print(f"Private Exponent: {d}")
print(f"Plaintext:        '{message}' (as integer: {m_int})")
print(f"Ciphertext:       {c_int}")
print(f"Decrypted:        '{decrypted_msg}'")
assert decrypted_msg == message



Modulus N:        1000000016000000063


Public Exponent:  65537


Private Exponent: 648946405777194593


Plaintext:        'HELLO RSA' (as integer: 1333159023296663147329)


Ciphertext:       598758114184472090


Decrypted:        'HELLO RSA'
